In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

# load merged raw counts
project_root = Path.cwd()
counts_path = project_root / "quants" / "GSE60450_merged_raw_counts.txt"
counts = pd.read_csv(counts_path, sep="\t")

# keep sample columns only, then transpose so rows are samples and columns are genes
count_matrix = counts.drop(columns=["GeneID", "gene_name"], errors="ignore")
count_matrix = count_matrix.set_axis(count_matrix.columns.astype(str), axis=1)
count_matrix = count_matrix.T
count_matrix.index.name = "sample"

# log transform counts to reduce dominance of very highly expressed genes
count_matrix_log = count_matrix.apply(pd.to_numeric, errors="coerce").fillna(0)
count_matrix_log = count_matrix_log.loc[:, count_matrix_log.var(axis=0) > 0]
count_matrix_log = count_matrix_log.apply(lambda col: pd.to_numeric(col, errors="coerce"))
count_matrix_log = count_matrix_log.fillna(0)
count_matrix_log = np.log2(count_matrix_log + 1)

# sample metadata from the DESeq2 notebook
metadata = pd.DataFrame({
    "sample": [
        "GSM1480291", "GSM1480292",
        "GSM1480293", "GSM1480294",
        "GSM1480295", "GSM1480296",
        "GSM1480297", "GSM1480298",
        "GSM1480299", "GSM1480300",
        "GSM1480301", "GSM1480302",
    ],
    "cell_type": [
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Luminal", "Luminal",
        "Basal", "Basal",
        "Basal", "Basal",
        "Basal", "Basal",
    ],
    "stage": [
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
        "virgin", "virgin",
        "18.5dP", "18.5dP",
        "2dL", "2dL",
    ],
    "replicate": [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2],
}).set_index("sample")

# run PCA
pca = PCA(n_components=2)
pcs = pca.fit_transform(count_matrix_log)
pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=count_matrix_log.index)
pca_df = pca_df.join(metadata)

# plot PCA
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cell_type",
    style="stage",
    s=120,
    ax=ax,
)

for sample, row in pca_df.iterrows():
    ax.text(row["PC1"] + 0.15, row["PC2"] + 0.15, sample, fontsize=9)

ax.set_title("PCA of GSE60450 raw counts")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

pca_df
